# Kommuner contribution

Publishes Norwegian municipality polygons (357 features) to the `contributions` blob storage container as `municipalities.parquet`.

**Source:** [Administrative enheter — kommuner (Geonorge)](https://kartkatalog.geonorge.no/metadata/administrative-enheter-kommuner/041f1e6e-bdbc-4091-b48f-8a5990f3cc5b). The notebook downloads the `Basisdata_0000_Norge_4258_Kommune_GeoJSON` zip from Geonorge's `nedlasting` host and extracts the `.geojson` to `data/input/` automatically.

**Output schema:** `region` (kommunenummer), `name` (kommunenavn), `wkb` (geometry as WKB binary). The `region` column mirrors `counties.parquet` so doppa-data spatial-join entrypoints need no schema changes.

**CRS note:** Source declares EPSG:4258 (ETRS89 geographic). Coordinate difference vs. EPSG:4326 is sub-meter for Norway, so the output is labeled as EPSG:4326 to match the buildings datasets — no reprojection is performed.

In [1]:
from io import BytesIO

import geopandas as gpd
from shapely import to_wkb

from src import Config
from src.persistence import create_duckdb_context, create_blob_storage_context
from src.utils import download_kommune_geojson_file, upload_parquet_to_container

In [2]:
KOMMUNE_ZIP_URL = "https://nedlasting.geonorge.no/geonorge/Basisdata/Kommuner/GeoJSON/Basisdata_0000_Norge_4258_Kommuner_GeoJSON.zip"
input_path = Config.INPUT_DATA_DIR / "Basisdata_0000_Norge_4258_Kommune_GeoJSON.geojson"

download_kommune_geojson_file(zip_url=KOMMUNE_ZIP_URL, target_path=input_path)

## Load GeoJSON and build output dataframe

The Geonorge file is UTF-8 with BOM. We read the raw bytes, strip the BOM, then hand a `BytesIO` to GeoPandas so GDAL never sees the BOM.

In [3]:
raw_bytes = input_path.read_bytes()
if raw_bytes.startswith(b"\xef\xbb\xbf"):
    raw_bytes = raw_bytes[3:]

kommuner = gpd.read_file(BytesIO(raw_bytes))
kommuner = kommuner.set_crs(epsg=4326, allow_override=True)

kommuner = kommuner.rename(columns={"kommunenummer": "region", "kommunenavn": "name"})
kommuner = kommuner[["region", "name", "geometry"]]
kommuner["wkb"] = kommuner["geometry"].apply(to_wkb)
kommuner = kommuner[["region", "name", "wkb"]]

In [4]:
output_path = str(Config.OUTPUT_DATA_DIR / "municipalities.parquet")

db = create_duckdb_context()
db.register("kommuner", kommuner)
db.execute(
    f"""
    COPY (
        SELECT region, name, wkb
        FROM kommuner
    ) TO '{output_path}'
    """
)

## Upload parquet file to Azure Blob Storage

In [5]:
storage_client = create_blob_storage_context()
upload_parquet_to_container(
    blob_service_client=storage_client,
    parquet_path=output_path,
    blob_name="municipalities.parquet",
    container_name="contributions"
)

## Smoke test: read uploaded blob back and assert row count

In [6]:
row_count = db.sql(
    "SELECT COUNT(*) AS n FROM read_parquet('az://contributions/municipalities.parquet')"
).fetchone()[0]

assert row_count == 357, f"Expected 357 municipalities, got {row_count}"
row_count

357